# Dunnhumby M5_A·M5_B — 남은8시드 / 최종10시드 집계

**M5_A = 확정 적응형 M2 + M3 / M5_B = 확정 적응형 M2 + 고객 내 평균1 M4.**

- 새 학습: seed **42,43,44,45,46,47,48,51**, 각 시드에서 **A → B** 순차 실행(총16회).
- 재사용: A/B seed49·50 4개 + M1/M2/M3/M4/M5_C seed42~51 50개. **참조 재학습·재평가 없음.**
- 공통 데이터·특징·그래프는 한 번만 준비. 각 새 모형은 새 초기값에서 하나의 optimizer로 공동 학습.
- 학습 DAY1~704 / test DAY705~711, 고정300epoch 마지막 점 한 번 평가. validation·holdout·조기종료 없음.
- CLV365일·ID64·2층·batch8192·lr0.0005·ID L2 0.001·uniform 음성1·MIN_ITEM_INTER=1 유지. 학습 구매쌍은 정답·후보에서 제외.
- 전체70개(7모형×10시드)가 모였을 때만 **10시드 산술평균·표본표준편차(ddof=1)** 및 대응 비교를 출력합니다. 일부만 완료되면 부분 결과로 저장합니다.
- 이미 관찰한 test의 반복 비교입니다. 표준편차는 시드 변동이며 신뢰구간이 아닙니다. 유의성·새 독립 확증·CLV 배정 귀속은 주장하지 않습니다.

런타임 유형을 **L4 GPU**로 선택하고 위에서부터 실행하세요. 중단 후 같은 노트북·경로로 재실행하면 완료 모델은 건너뛰고 저장 epoch부터 재개합니다. 기존 실행 폴더는 변경하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
# Errno107(Drive 연결 끊김)이 나오면, 학습이 멈춘 상태에서 다음 줄을 실행한 뒤 준비 셀부터 재실행:
# drive.mount('/content/drive', force_remount=True)
assert torch.cuda.is_available(), 'L4 GPU를 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 {GPU}입니다. L4로 변경하세요.'
SOURCE_COMMIT = 'a96eadb915c027599bf8b23c41eb34bb34d70d6e'
REPO = Path('/content/clv-dh-m5-ab-final10-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT], check=True)
sys.path.insert(0,str(REPO))
print('GPU:',GPU,'| 고정 코드:',SOURCE_COMMIT)

## 1. 기존 원본 확인·공통 입력 준비 (학습 없음)
아래 네 참조는 기존 Drive 폴더에서 자동으로 찾습니다. 원본과 다르거나 파일이 없으면 **학습 전에 멈추며 대신 학습하지 않습니다.**

필요하면 보관한 원본 ZIP을 Drive에 놓고 해당 경로만 지정하세요:

- `clv_m2_adaptive_dunnhumby_final10_results.zip`
- `clv_m4_user_centered_final10_results.zip`
- `dh_m5_ab_adaptive_lastweek_seed49_results.zip`
- `dh_m5_ab_adaptive_lastweek_seed50_results.zip`

이전 시드에서 준비 도중 Drive가 끊겼던 오류는 모델 오류와 다릅니다. 연결 복구 후 이 셀을 다시 실행하세요. 재개 시 최초 참조·출력 경로를 유지하세요.

In [ ]:
import clv_dh_m5_ab_adaptive_remaining8 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_dunnhumby_final10_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_m4_user_centered_final10_results.zip',
    # seed49_reference='/content/drive/MyDrive/.../dh_m5_ab_adaptive_lastweek_seed49_results.zip',
    # seed50_reference='/content/drive/MyDrive/.../dh_m5_ab_adaptive_lastweek_seed50_results.zip',
)
print(json.dumps(screen.asdict(cfg),ensure_ascii=False,indent=2))
prepared = screen.prepare(cfg)
print(json.dumps(screen.report(prepared)['reading'],ensure_ascii=False,indent=2))

## 2. 남은8시드 A→B 순차 학습
총16개의 모델 중 완료된 모델은 건너뜁니다. 미완료 모델은 epoch checkpoint에서 재개합니다.
각 모델300epoch 종료 후 test 평가·결과 저장을 마치고 다음 모델로 넘어갑니다.
학습 loss와 test 성능을 혼동하지 말고, 학습 중 다른 셀을 실행하지 마세요.

In [ ]:
result = screen.run(prepared)
assert result['complete']
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))

## 3. 결과 확인 — 전체10시드 완료 후 평균±표준편차
모든 모형에 **동일한 seed42~51**을 사용합니다. M1/M2/M3/M4/M5_C는 별도 실행의 동일조건 저장값입니다.
`summary.csv`는 전체142지표의 평균·표준편차·최솟값·최댓값을 보존합니다.
`comparison.csv`는 A/B와 다른6모형의 평균비율 변화 및 대응 차이·표준편차·상승/하락 시드 수를 담습니다.
`paired_comparison.csv`는 각시드의 원값과 차이를 보존합니다. 양수는 수치 증가이며 모든 노출지표에서 개선을 뜻하지 않습니다.
중단 후 준비 셀을 다시 실행하고 이 셀만 실행하면 **저장된 완료 결과만** 읽습니다. 추가 학습·test 평가는 없습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))
if result['complete']:
    print('10시드 평균 ± 표본표준편차 (신뢰구간 아님)')
    display(result['core_mean_std'])
    comparison = result['comparison']
    view = comparison[(comparison['reference']=='M1') & comparison['metric'].isin(screen.CORE)]
    display(view[['model','metric','ratio_of_means_pct','paired_delta_mean','paired_delta_std',
                  'positive_delta_seeds','negative_delta_seeds','tie_seeds']])
    print('평균비율 변화(%) = 100 × (후보10시드평균 / 참조10시드평균 − 1). 유의성 주장 없음.')
else:
    print('10시드 미완료 — 일부 시드 평균을 최종 평균으로 표시하지 않습니다.')
    display(result['absolute'][['model','seed','source'] + list(screen.CORE)])

## 4. 결과 ZIP 다운로드
전체 완료라면 최종10시드 ZIP, 미완료라면 이름에 `partial`이 붙은 부분 ZIP을 받습니다.
학습 중에는 실행하지 마세요. 원본 결과·진단·protocol·새 학습이력과 기존49/50 사본이 함께 포함됩니다.

In [ ]:
from google.colab import files
result = screen.report(prepared)
suffix = 'final10' if result['complete'] else 'partial'
output = screen.export_zip(prepared, f'/content/dh_m5_ab_adaptive_lastweek_{suffix}_results.zip',
                           allow_partial=not result['complete'])
files.download(output)